In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.pyplot as plt

: 

In [ ]:
# 1. Set up the GPU device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# 2. Load the combined data
df = pd.read_csv('data/combined_training_data.csv')

# --- FOOLPROOF CLEANING ---
# Strip whitespace from all column names
df.columns = df.columns.str.strip()

# Force the label column to be a string, then strip all spaces
df['label'] = df['label'].astype(str).str.strip()
# --------------------------

# 3. Separate features (X) and labels (y)
X = df.drop(columns=['label', 'time']).values 
y_raw = df['label'].values

# 4. Encode the text labels into numbers
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
num_classes = len(label_encoder.classes_)
print(f"Classes found: {label_encoder.classes_}")

# 5. Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Scale the data (Standardization)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 7. Convert everything into PyTorch Tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

In [ ]:
# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create data loaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class SensorClassifier(nn.Module):
    def __init__(self, input_size, num_classes):
        super(SensorClassifier, self).__init__()
        # Layer 1
        self.layer1 = nn.Linear(input_size, 64)
        self.relu1 = nn.ReLU()
        # Layer 2
        self.layer2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        # Output Layer
        self.output_layer = nn.Linear(32, num_classes)

    def forward(self, x):
        out = self.layer1(x)
        out = self.relu1(out)
        out = self.layer2(out)
        out = self.relu2(out)
        out = self.output_layer(out)
        return out

# Instantiate the model
input_size = X_train_tensor.shape[1] # Number of columns/features
model = SensorClassifier(input_size, num_classes).to(device)
print("Model loaded to:", next(model.parameters()).device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50
train_losses = []

print("Starting training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        # Send data batches to the GPU
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')

In [ ]:
model.eval()
torch.save(model, 'model.pth')
torch.save(scaler, 'scaler.pt')
torch.save(label_encoder, 'encoder.pt')


correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        # Send test data batches to the GPU
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy on the test set: {accuracy:.2f}%')

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), train_losses, label='Training Loss', color='blue', linewidth=2)
plt.title('Model Training Loss Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# 1. Define the path to your new test CSV
new_test_file = 'data/anish-scissors-1_50hz_dataset.csv' 
new_test_df = pd.read_csv(new_test_file)

In [ ]:
# --- NEW: Print the exact label mapping ---
print("--- Model's Known Label Mapping ---")
for num, text_label in enumerate(label_encoder.classes_):
    print(f"  {num}  ->  '{text_label}'")
print("-----------------------------------\n")
# ------------------------------------------



# 2. Clean the column names and labels
new_test_df.columns = new_test_df.columns.str.strip()
if new_test_df['label'].dtype == 'object':
    new_test_df['label'] = new_test_df['label'].str.strip()

# 3. Safeguard: Filter out unknown labels and print them
known_classes = set(label_encoder.classes_)
valid_rows = new_test_df['label'].isin(known_classes)

if not valid_rows.all():
    unknown_labels = set(new_test_df['label']) - known_classes
    dropped_count = len(new_test_df) - valid_rows.sum()
    print(f"⚠️ Warning: Found unknown labels in new data: {unknown_labels}")
    print(f"  -> Dropping {dropped_count} rows that the model wasn't trained on.\n")
    new_test_df = new_test_df[valid_rows]

if len(new_test_df) == 0:
    raise ValueError("No valid labels left to test! Check your CSV labels.")

# 4. Separate features (X) and true labels (y)
X_new = new_test_df.drop(columns=['label', 'time']).values 
y_new_raw = new_test_df['label'].values

# 5. Preprocess using the ALREADY FITTED scaler and encoder
y_new = label_encoder.transform(y_new_raw)
X_new_scaled = scaler.transform(X_new)

# 6. Convert to PyTorch Tensors
X_new_tensor = torch.FloatTensor(X_new_scaled)
y_new_tensor = torch.LongTensor(y_new)

# 7. Create the DataLoader
new_test_dataset = TensorDataset(X_new_tensor, y_new_tensor)
new_test_loader = DataLoader(new_test_dataset, batch_size=64, shuffle=False)

# 8. Evaluate the model
model.eval() 
correct_new = 0
total_new = 0

print(f"Testing on {len(new_test_df)} valid samples...\n")

with torch.no_grad():
    for inputs, labels in new_test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        
        total_new += labels.size(0)
        correct_new += (predicted == labels).sum().item()

new_accuracy = 100 * correct_new / total_new
print(f'Done! Accuracy on the unseen test set: {new_accuracy:.2f}%')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Gather all predictions in order
model.eval()
all_predictions = []

with torch.no_grad():
    for inputs, labels in new_test_loader:
        # Send to GPU to process
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        
        # Bring the predictions back to the CPU so Pandas and Matplotlib can use them
        all_predictions.extend(predicted.cpu().numpy())

# 2. Add the results back into your original dataframe
new_test_df['predicted_class'] = all_predictions
# Convert the number predictions (e.g., 0, 1) back into text labels ('fist', 'wave')
new_test_df['predicted_label'] = label_encoder.inverse_transform(all_predictions)
# Create a True/False column for whether the prediction was right
new_test_df['is_correct'] = new_test_df['predicted_label'] == new_test_df['label']

# 3. Create the stacked time graphs
fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
time_vals = new_test_df['time']

# We will plot the colors over your 's1_X' data so the graph has a recognizable shape
sensor_vals = new_test_df['s1_X'] 

# --- Graph 1: Colored by Model Prediction ---
ax1 = axes[0]
classes = label_encoder.classes_
# Grab a distinct color map for however many classes you have
cmap = plt.get_cmap('tab10') 

for i, cls in enumerate(classes):
    # Mask to only grab the rows where the model guessed this specific class
    mask = new_test_df['predicted_label'] == cls
    ax1.scatter(time_vals[mask], sensor_vals[mask], label=f'Pred: {cls}', color=cmap(i), s=15)

ax1.set_title('Model Predictions Over Time (Overlayed on s1_X)', fontsize=14, fontweight='bold')
ax1.set_ylabel('s1_X Amplitude', fontsize=12)
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_facecolor('#f8f9fa')

# --- Graph 2: Colored by Correct / Incorrect ---
ax2 = axes[1]
correct_mask = new_test_df['is_correct'] == True
incorrect_mask = new_test_df['is_correct'] == False

# Plot Correct predictions as Green, Incorrect as Red
ax2.scatter(time_vals[correct_mask], sensor_vals[correct_mask], label='Correct', color='#2ecc71', s=15)
ax2.scatter(time_vals[incorrect_mask], sensor_vals[incorrect_mask], label='Incorrect', color='#e74c3c', s=15)

ax2.set_title('Prediction Accuracy (Green = Correct, Red = Incorrect)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_ylabel('s1_X Amplitude', fontsize=12)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_facecolor('#f8f9fa')

plt.tight_layout()
plt.show()